# KinoCompass – смесь «популярность × Mult-VAE» на GPU

Гибрид шага 6б: популярные за последний год смешиваются с Mult-VAE (лучший вариант сетки шага 6а: «вкус» 400, β 0,5) по рангам с весом популярности 0,2; 0,4; 0,6; 0,8 (`model/hybrid.py`). Mult-VAE обучается один раз на все веса. Пачки для обучения собираются прямо на GPU – эпоха быстрее, чем в первом прогоне (25 с).

Ноутбук клонирует репозиторий и запускает **тот же** код проверки, что и в Docker (`model/evaluate.py`): те же люди, сценарии и метрики – смесь сравнивается с «популярность × ALS» парным тестом.

**Как запустить:** «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU**, затем «Среда выполнения» → «Выполнить все». В конце скачается архив `hybrid_vae_<период>.zip` с отчётом – его нужно положить в `model/reports/`.

Время на T4 (оценка): загрузка набора и сценарии – около 3 минут, обучение Mult-VAE – 5–15 минут, проверка четырёх смесей – 5–10 минут. Вкладку не закрывать до конца.

In [ ]:
# проверочная часть: validation – выбор настроек; test – только на шаге 6, один раз
PERIOD = "validation"

In [ ]:
import torch

assert torch.cuda.is_available(), "нет GPU: Среда выполнения -> Сменить среду выполнения -> T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "ГБ")

In [ ]:
# из /content: при повторном запуске в том же сеансе текущая папка – внутри удаляемого клона
%cd /content
!rm -rf /content/KinoCompass && git clone -q --depth 1 https://github.com/AlexTrav/KinoCompass.git /content/KinoCompass
!git -C /content/KinoCompass log --oneline -1
# те же версии, что в образе Docker (torch в Colab уже есть)
!pip install -q requests==2.34.2 pandas==3.1.0 pyarrow==26.0.1 scipy==1.18.1

In [ ]:
%cd /content/KinoCompass/model
!PYTHONPATH=/content/KinoCompass/data:/content/KinoCompass/research PERIOD={PERIOD} python -W ignore evaluate.py hybrid_vae

In [ ]:
from google.colab import files

!cd /content/KinoCompass/model && zip -q -r /content/hybrid_vae_{PERIOD}.zip reports/hybrid_vae && ls -la reports/hybrid_vae
files.download(f"/content/hybrid_vae_{PERIOD}.zip")